In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
scaler = joblib.load("../models/scaler.joblib")
final_model = joblib.load("../models/final_model.joblib")
threshold = joblib.load("../models/threshold.joblib")
diagnosis_models = joblib.load("../models/diagnosis_models.joblib")

print("Threshold:", threshold)
print("Diagnosis models:", list(diagnosis_models.keys()))

Threshold: 0.25000000000000006
Diagnosis models: ['TWF', 'HDF', 'PWF', 'OSF']


In [3]:
feature_cols = list(pd.read_csv("../data/processed/X_train.csv", nrows=1).columns)
feature_cols

['quality_type',
 'machine_type',
 'air_temp',
 'process_temp',
 'rpm',
 'torque',
 'cutting_force',
 'flank_wear',
 'temp_diff',
 'power',
 'wear_force']

In [4]:
def build_features(quality_type, machine_type, air_temp, process_temp,
                   rpm, torque, cutting_force, flank_wear):

    row = {
        "quality_type": {"L": 0, "M": 1, "H": 2}[quality_type],
        "machine_type": {"Milling": 0, "Turning": 1}[machine_type],
        "air_temp": air_temp,
        "process_temp": process_temp,
        "rpm": rpm,
        "torque": torque,
        "cutting_force": cutting_force,
        "flank_wear": flank_wear,
        "temp_diff": process_temp - air_temp,
        "power": torque * rpm * 2 * np.pi / 60,
        "wear_force": flank_wear * cutting_force,
    }

    return pd.DataFrame([row])[feature_cols]

In [5]:
def get_risk_level(prob):
    if prob >= 0.60:
        return "High"
    elif prob >= 0.30:
        return "Medium"
    else:
        return "Low"

In [6]:
OSF_LIMIT = {"L": 1200, "M": 1320, "H": 1600}


def get_recommendations(quality_type, machine_type, air_temp, process_temp,
                        rpm, torque, cutting_force, flank_wear):
    advice = []

    temp_diff = process_temp - air_temp
    power = torque * rpm * 2 * np.pi / 60
    wear_force = flank_wear * cutting_force

    if flank_wear >= 0.30:
        advice.append("Flank wear has reached the 0.30 mm limit: replace the tool.")

    if temp_diff < 8.6 and rpm < 1380:
        advice.append("Low temperature difference at low speed: check the cooling system.")

    if power < 3500:
        advice.append("Power is too low: check for slipping or an unstable load.")
    elif power > 9000:
        advice.append("Power is too high: reduce the load or speed.")

    if wear_force > OSF_LIMIT[quality_type]:
        advice.append("Overstrain risk: reduce the cutting force (lower feed or depth of cut) or replace the worn tool.")

    if not advice:
        advice.append("All readings are in the normal range. Continue regular monitoring.")

    return advice

In [7]:
def predict_machine(quality_type, machine_type, air_temp, process_temp,
                    rpm, torque, cutting_force, flank_wear):

    features = build_features(quality_type, machine_type, air_temp, process_temp,
                              rpm, torque, cutting_force, flank_wear)
    scaled = pd.DataFrame(scaler.transform(features), columns=feature_cols)

    prob = float(final_model.predict_proba(scaled)[0, 1])

    type_probs = {
        t: float(diagnosis_models[t].predict_proba(scaled)[0, 1])
        for t in diagnosis_models
    }

    return {
        "failure_probability": round(prob, 3),
        "failure_alert": prob >= threshold,
        "risk_level": get_risk_level(prob),
        "likely_cause": max(type_probs, key=type_probs.get),
        "cause_probabilities": {k: round(v, 3) for k, v in type_probs.items()},
        "advice": get_recommendations(quality_type, machine_type, air_temp, process_temp,
                                      rpm, torque, cutting_force, flank_wear),
    }

Test

In [9]:
predict_machine("L", "Milling", 298, 308.6, 1500, 40, 2600, 0.05)

{'failure_probability': 0.0,
 'failure_alert': np.False_,
 'risk_level': 'Low',
 'likely_cause': 'PWF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 0.0, 'PWF': 0.0, 'OSF': 0.0},
 'advice': ['All readings are in the normal range. Continue regular monitoring.']}

In [10]:
predict_machine("L", "Milling", 298, 308.6, 1400, 60, 4600, 0.35)

{'failure_probability': 1.0,
 'failure_alert': np.True_,
 'risk_level': 'High',
 'likely_cause': 'OSF',
 'cause_probabilities': {'TWF': 0.008, 'HDF': 0.0, 'PWF': 0.0, 'OSF': 1.0},
 'advice': ['Flank wear has reached the 0.30 mm limit: replace the tool.',
  'Overstrain risk: reduce the cutting force (lower feed or depth of cut) or replace the worn tool.']}

In [11]:
predict_machine("M", "Turning", 302, 310, 1300, 45, 1600, 0.10)

{'failure_probability': 0.998,
 'failure_alert': np.True_,
 'risk_level': 'High',
 'likely_cause': 'HDF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 1.0, 'PWF': 0.0, 'OSF': 0.0},
 'advice': ['Low temperature difference at low speed: check the cooling system.']}

In [12]:
predict_machine("H", "Milling", 298, 308.6, 1300, 70, 4700, 0.10)

{'failure_probability': 1.0,
 'failure_alert': np.True_,
 'risk_level': 'High',
 'likely_cause': 'PWF',
 'cause_probabilities': {'TWF': 0.0, 'HDF': 0.001, 'PWF': 1.0, 'OSF': 0.004},
 'advice': ['Power is too high: reduce the load or speed.']}

In [13]:
raw = pd.read_csv("../data/ai4i2020_augmented.csv")
raw = raw.rename(columns={
    "Quality Type": "quality_type",
    "Machine Type": "machine_type",
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Cutting force [N]": "cutting_force",
    "Tool flank wear VB [mm]": "flank_wear",
    "Machine failure": "failure"
})

failed = raw[raw["failure"] == 1].head(5)

for _, r in failed.iterrows():
    result = predict_machine(r["quality_type"], r["machine_type"], r["air_temp"],
                             r["process_temp"], r["rpm"], r["torque"],
                             r["cutting_force"], r["flank_wear"])
    print(r["machine_type"], "| prob:", result["failure_probability"],
          "| risk:", result["risk_level"], "| cause:", result["likely_cause"])

Milling | prob: 0.999 | risk: High | cause: PWF
Milling | prob: 1.0 | risk: High | cause: PWF
Milling | prob: 0.994 | risk: High | cause: OSF
Turning | prob: 0.98 | risk: High | cause: TWF
Milling | prob: 1.0 | risk: High | cause: OSF
